In [ ]:
import numpy as np
import pandas as pd
import os

# --- SETTINGS ---
# We assume the files are uploaded to Colab's main directory (left sidebar).
BASE_DIR = "."

# PREDEFINED WEIGHTS (UnixCoder Dominant)
ensemble_config = [
    # (File Name, Weight)
    ("submission_UnixCoder_P2_Best_probs.npy",  0.43),  # Champion Model
    ("submission_UnixCoder_P3_Final_probs.npy", 0.23),
    ("submission_GCB_P3_Final_probs.npy",        0.34),
    ("submission_GCB_P2_Best_probs.npy",        0.0),

]

print("🚀 Starting Ensemble (Colab Directory)...")

final_probs = None
total_weight = 0

# 1. Load and Weight Probabilities
for filename, weight in ensemble_config:
    filepath = os.path.join(BASE_DIR, filename)

    if os.path.exists(filepath):
        print(f"➕ Adding: {filename} (Weight: {weight})")
        probs = np.load(filepath)

        if final_probs is None:
            final_probs = probs * weight
        else:
            final_probs += probs * weight

        total_weight += weight
    else:
        print(f"⚠️ ERROR: File not found -> {filename}")
        print("   (Make sure the file is uploaded in the left sidebar)")

print(f"📊 Total Applied Weight: {total_weight:.2f}")

# 2. Select Class with Highest Probability
if final_probs is not None:
    predictions = np.argmax(final_probs, axis=1)

    # 3. Save as CSV
    # test.parquet should also be uploaded in the sidebar
    test_path = os.path.join(BASE_DIR, "test.parquet")

    if os.path.exists(test_path):
        test_df = pd.read_parquet(test_path)
        ids = test_df['ID'] if 'ID' in test_df.columns else test_df.index
    else:
        print("⚠️ 'test.parquet' not found! IDs will be auto-generated starting from 0.")
        ids = range(len(predictions))

    output_path = "submission_ENSEMBLE_UnixDominant-9.csv"

    submission = pd.DataFrame({'ID': ids, 'label': predictions})
    submission.to_csv(output_path, index=False)

    print(f"\n✅ Ensemble completed! File created: {output_path}")
    print(f"   (Right-click the file in the left sidebar to download it)")
else:
    print("\n❌ No files could be loaded. Please check that the .npy files are uploaded to Colab.")
